In [13]:
import pandas as pd

In [14]:
dataset_fps = ['aneesh_labels.csv', 'rishaant_labels.csv', 'josh_labels.csv', 'sharon_labels.csv']
dataset_fps = ['camera_data/' + fp for fp in dataset_fps]

replace_dataset = 'camera_data/majority_replaced_labels.csv'

In [15]:
def get_most_common_choice(dataset_fps, column, replace_fp):
    datasets = [pd.read_csv(dataset_fp).drop_duplicates(subset=['image']).set_index('image') for dataset_fp in dataset_fps]

    column_choices = pd.concat([dataset[column] for dataset in datasets], axis = 1, ignore_index = False)



    most_common_choices = column_choices.mode(axis = 1)[0]

    assert len(datasets[0]) == len(most_common_choices)

    return_dataset = datasets[0].copy()

    return_dataset[column] = most_common_choices

    replacement_choices = pd.read_csv(replace_fp).set_index('image')[column]

    replacement_choices = replacement_choices[replacement_choices.notna()]

    return_dataset.loc[replacement_choices.index, column] = replacement_choices

    return_dataset.reset_index(inplace=True)
    assert len(return_dataset) == len(datasets[0])

    return return_dataset


In [16]:
import requests
datasets = [
    "/sanitized-datasets/3c72418d-4248-5130-952c-aa43f87a7a8f", #Coronado Hills data
    "/sanitized-datasets/a405359a-1619-513b-b7ef-a472e3d4f131" #All data
  ]

data = []
for dataset in datasets:
    request = requests.get(f'https://tools.alertcalifornia.org{dataset}').json()
    data.append(pd.DataFrame(request['frames']))

get_id = lambda url: url.split('/')[-1]

In [17]:
data[1]['camera_id'].value_counts()

camera_id
Axis-MesaGrandeNorth    3113
Axis-CoronadoHillsS     2334
Axis-PalomarObs1        2044
Axis-ToroPeak1           512
Axis-Dewdrop1            424
Axis-Berryessa           404
Name: count, dtype: int64

In [18]:
majority_labels = get_most_common_choice(dataset_fps, 'choice', replace_dataset)


camera_ids = data[1][['camera_id', 'url']].drop_duplicates()
camera_ids['url'] = camera_ids['url'].apply(lambda x: 'https://tools.alertcalifornia.org' + x)

majority_labels = majority_labels.merge(camera_ids, left_on='image', right_on = 'url', how = 'left').drop(columns = ['url']).rename(columns = {'camera_id' : 'location'})

majority_labels.to_csv('camera_data/majority_labels.csv', index = False)

coronado_urls = data[1][data[1]['url'].apply(get_id).isin(data[0]['url'].apply(get_id))]['url'].apply(lambda x: 'https://tools.alertcalifornia.org' + x)


coronado_labels = majority_labels[majority_labels['image'].isin(coronado_urls)]

coronado_labels_to_inject = coronado_labels[['image', 'choice']]
coronado_labels_to_inject['image'] = coronado_labels_to_inject['image'].apply(get_id)

coronado_labels_to_inject.columns = ['id', 'choice']

coronado_labels_to_inject.to_csv('camera_data/coronado_labels_to_inject.csv', index = False)
coronado_labels.to_csv('camera_data/coronado_hills_data.csv', index = False)

In [19]:
majority_labels

,image,annotation_id,annotator,choice,created_at,id,lead_time,ptz,updated_at,location
0,https://tools.alertcalifornia.org/sanitized-da...,94414,11,3 (Major),2026-05-19T22:12:45.236709Z,1339752,6.215,"{""pan"":174.02000427246094,""tilt"":-0.1700000017...",2026-05-19T22:12:45.236728Z,Axis-CoronadoHillsS
1,https://tools.alertcalifornia.org/sanitized-da...,80743,11,2 (Minor),2026-04-22T16:28:53.923282Z,1339753,3.317,"{""pan"":174.02000427246094,""tilt"":-0.1700000017...",2026-04-22T16:28:53.923316Z,Axis-CoronadoHillsS
2,https://tools.alertcalifornia.org/sanitized-da...,81046,11,1 (Zero),2026-04-22T17:25:03.446452Z,1339754,1.884,"{""pan"":174.02000427246094,""tilt"":-0.1700000017...",2026-04-22T17:25:03.446467Z,Axis-CoronadoHillsS
3,https://tools.alertcalifornia.org/sanitized-da...,94495,11,2 (Minor),2026-05-19T22:23:30.802091Z,1339755,10.188,"{""pan"":174.02000427246094,""tilt"":-0.1700000017...",2026-05-19T22:23:30.802107Z,Axis-CoronadoHillsS
4,https://tools.alertcalifornia.org/sanitized-da...,90500,11,2 (Minor),2026-05-18T22:16:08.142174Z,1339756,2.944,"{""pan"":174.02000427246094,""tilt"":-0.1700000017...",2026-05-18T22:16:08.142194Z,Axis-CoronadoHillsS
...,...,...,...,...,...,...,...,...,...,...
6946,https://tools.alertcalifornia.org/sanitized-da...,94885,11,1 (Zero),2026-05-20T00:04:31.630279Z,1346698,4.430,"{""pan"":278.17999267578125,""tilt"":-0.3300000131...",2026-05-20T00:04:31.630298Z,Axis-Dewdrop1
6947,https://tools.alertcalifornia.org/sanitized-da...,77858,11,1 (Zero),2026-04-11T23:17:30.694968Z,1346699,1.226,"{""pan"":278.17999267578125,""tilt"":-0.3300000131...",2026-04-11T23:17:30.694974Z,Axis-Dewdrop1
6948,https://tools.alertcalifornia.org/sanitized-da...,87828,11,1 (Zero),2026-05-15T18:55:44.460179Z,1346700,2.408,"{""pan"":278.17999267578125,""tilt"":-0.3300000131...",2026-05-15T18:55:44.460194Z,Axis-Dewdrop1
6949,https://tools.alertcalifornia.org/sanitized-da...,90476,11,2 (Minor),2026-05-18T22:14:11.386139Z,1346701,3.913,"{""pan"":278.17999267578125,""tilt"":-0.3300000131...",2026-05-18T22:14:11.386162Z,Axis-Dewdrop1


In [21]:
counts_pivot = pd.pivot_table(majority_labels, index = 'choice', columns = 'location', 
                              values = 'image', aggfunc = 'count', margins=True, margins_name='Total')
counts_pivot = counts_pivot[['Axis-CoronadoHillsS', 'Axis-MesaGrandeNorth', 'Axis-PalomarObs1', 
                             'Axis-ToroPeak1', 'Axis-Dewdrop1', 'Axis-Berryessa', 'Total']].fillna(0)

In [22]:
counts_pivot

location,Axis-CoronadoHillsS,Axis-MesaGrandeNorth,Axis-PalomarObs1,Axis-ToroPeak1,Axis-Dewdrop1,Axis-Berryessa,Total
choice,,,,,,,
1 (Zero),977,1367,1271,155,143,110,4023
2 (Minor),813,217,363,60,38,49,1540
3 (Major),226,91,254,26,24,9,630
4 (Complete),309,240,153,15,7,34,758
Total,2325,1915,2041,256,212,202,6951


In [23]:
counts_pivot.columns

Index(['Axis-CoronadoHillsS', 'Axis-MesaGrandeNorth', 'Axis-PalomarObs1',
       'Axis-ToroPeak1', 'Axis-Dewdrop1', 'Axis-Berryessa', 'Total'],
      dtype='str', name='location')

In [24]:
training_urls = data[1][~data[1]['camera_id'].isin(['Axis-ToroPeak1', 'Axis-Dewdrop1', 'Axis-Berryessa'])]['url'].apply(lambda x: 'https://tools.alertcalifornia.org' + x)

training_labels = majority_labels[majority_labels['image'].isin(training_urls)]
unseen_labels = majority_labels[~majority_labels['image'].isin(training_urls)]

training_labels.to_csv('camera_data/training_set_cameras_data.csv', index=False)
unseen_labels.to_csv('camera_data/unseen_set_cameras_data.csv', index=False)

In [25]:
coronado_labels_to_inject.head()

,id,choice
0,4bd763a1-79e2-5a17-9ca0-4c065b09f869.jpg,3 (Major)
1,73e15d36-e9df-5cea-a140-3a1d3856fc00.jpg,2 (Minor)
2,8bb77bee-06a0-55b3-a98c-a30ac4f5202c.jpg,1 (Zero)
3,37142b8a-dea1-5b5c-a210-c99aa4273666.jpg,2 (Minor)
4,c4f156db-b801-5ef7-93d2-a410257c5efe.jpg,2 (Minor)
